# OR Dataset Exploration

This notebook explores the Oregon (OR) dataset to understand household counts, income distribution, and demographic characteristics.

In [1]:
from policyengine_us import Microsimulation
import pandas as pd
import numpy as np

OR_DATASET = "hf://policyengine/policyengine-us-data/states/OR.h5"

In [2]:
# Load OR dataset
sim = Microsimulation(dataset=OR_DATASET)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


OR.h5:   0%|          | 0.00/52.9M [00:00<?, ?B/s]

In [3]:
# Check dataset size
household_weight = sim.calculate("household_weight", period=2025)
household_count = sim.calculate("household_count", period=2025, map_to="household")
person_count = sim.calculate("person_count", period=2025, map_to="household")

print(f"Number of households in dataset: {len(household_weight):,}")
print(f"Household count (weighted): {household_count.sum():,.0f}")
print(f"Person count (weighted): {person_count.sum():,.0f}")

Number of households in dataset: 33,208
Household count (weighted): 1,421,300
Person count (weighted): 4,254,669


In [4]:
# Median household income and per capita income
household_net_income = sim.calculate("household_net_income", period=2025)
print(f"Median household income: ${household_net_income.median():,.0f}")

# Per capita income
total_income = household_net_income.sum()
total_persons = person_count.sum()
per_capita_income = total_income / total_persons
print(f"Per capita income: ${per_capita_income:,.0f}")

Median household income: $45,327
Per capita income: $20,320


In [5]:
# Check households with children
is_child = sim.calculate("is_child", period=2025, map_to="person")
household_id = sim.calculate("household_id", period=2025, map_to="person")
household_weight = sim.calculate("household_weight", period=2025, map_to="person")

# Create DataFrame
df_households = pd.DataFrame({
    'household_id': household_id,
    'is_child': is_child,
    'household_weight': household_weight
})

# Count children per household
children_per_household = df_households.groupby('household_id').agg({
    'is_child': 'sum',
    'household_weight': 'first'
}).reset_index()

# Calculate weighted household counts
total_households_with_children = children_per_household[children_per_household['is_child'] > 0]['household_weight'].sum()
households_with_1_child = children_per_household[children_per_household['is_child'] == 1]['household_weight'].sum()
households_with_2_children = children_per_household[children_per_household['is_child'] == 2]['household_weight'].sum()
households_with_3plus_children = children_per_household[children_per_household['is_child'] >= 3]['household_weight'].sum()

print(f"\nHouseholds with children (weighted):")
print(f"  Total households with children: {total_households_with_children:,.0f}")
print(f"  Households with 1 child: {households_with_1_child:,.0f}")
print(f"  Households with 2 children: {households_with_2_children:,.0f}")
print(f"  Households with 3+ children: {households_with_3plus_children:,.0f}")


Households with children (weighted):
  Total households with children: 430,918
  Households with 1 child: 176,598
  Households with 2 children: 126,902
  Households with 3+ children: 127,418


In [6]:
# Check children by age groups
df = pd.DataFrame({
    "household_id": sim.calculate("household_id", map_to="person"),
    "tax_unit_id": sim.calculate("tax_unit_id", map_to="person"),
    "person_id": sim.calculate("person_id", map_to="person"),
    "age": sim.calculate("age", map_to="person"),
    "person_weight": sim.calculate("person_weight", map_to="person")
})

# Filter for children and apply weights
children_under_18_df = df[df['age'] < 18]
children_under_6_df = df[df['age'] < 6]
children_under_3_df = df[df['age'] < 3]

# Calculate weighted totals
total_children = children_under_18_df['person_weight'].sum()
children_under_6 = children_under_6_df['person_weight'].sum()
children_under_3 = children_under_3_df['person_weight'].sum()

print(f"\nChildren by age:")
print(f"  Total children under 18: {total_children:,.0f}")
print(f"  Children under 6: {children_under_6:,.0f}")
print(f"  Children under 3: {children_under_3:,.0f}")


Children by age:
  Total children under 18: 882,828
  Children under 6: 250,611
  Children under 3: 106,289


In [7]:
# Poverty rate
person_in_poverty = sim.calculate("person_in_poverty", period=2025)
poverty_rate = person_in_poverty.mean() * 100

print(f"\nPoverty Rate:")
print(f"  Overall poverty rate: {poverty_rate:.2f}%")
print(f"  Persons in poverty: {person_in_poverty.sum():,.0f}")


Poverty Rate:
  Overall poverty rate: 35.24%
  Persons in poverty: 1,499,212


In [8]:
# Create weighted summary table
weighted_summary_data = {
    'Metric': [
        'Household count (weighted)',
        'Person count (weighted)',
        'Median household income',
        'Per capita income',
        'Total households with children',
        'Households with 1 child',
        'Households with 2 children',
        'Households with 3+ children',
        'Total children under 18',
        'Children under 6',
        'Children under 3',
        'Poverty rate',
        'Persons in poverty'
    ],
    'Value': [
        f"{household_count.sum():,.0f}",
        f"{person_count.sum():,.0f}",
        f"${household_net_income.median():,.0f}",
        f"${per_capita_income:,.0f}",
        f"{total_households_with_children:,.0f}",
        f"{households_with_1_child:,.0f}",
        f"{households_with_2_children:,.0f}",
        f"{households_with_3plus_children:,.0f}",
        f"{total_children:,.0f}",
        f"{children_under_6:,.0f}",
        f"{children_under_3:,.0f}",
        f"{poverty_rate:.2f}%",
        f"{person_in_poverty.sum():,.0f}"
    ]
}

weighted_df = pd.DataFrame(weighted_summary_data)

print("\n" + "="*60)
print("OR DATASET SUMMARY - WEIGHTED (Population Estimates)")
print("="*60)
print(weighted_df.to_string(index=False))
print("="*60)

# Save table
weighted_df.to_csv('or_dataset_summary_weighted.csv', index=False)
print("\nSummary saved to: or_dataset_summary_weighted.csv")


OR DATASET SUMMARY - WEIGHTED (Population Estimates)
                        Metric     Value
    Household count (weighted) 1,421,300
       Person count (weighted) 4,254,669
       Median household income   $45,327
             Per capita income   $20,320
Total households with children   430,918
       Households with 1 child   176,598
    Households with 2 children   126,902
   Households with 3+ children   127,418
       Total children under 18   882,828
              Children under 6   250,611
              Children under 3   106,289
                  Poverty rate    35.24%
            Persons in poverty 1,499,212

Summary saved to: or_dataset_summary_weighted.csv


In [9]:
# Households with $0 income
agi_hh = np.array(sim.calculate("adjusted_gross_income", period=2025, map_to="household"))
weights = np.array(sim.calculate("household_weight", period=2025))

zero_income_mask = agi_hh == 0
zero_income_count = weights[zero_income_mask].sum()
total_households = weights.sum()

print("\n" + "="*70)
print("HOUSEHOLDS WITH $0 INCOME")
print("="*70)
print(f"Household count: {zero_income_count:,.0f}")
print(f"Percentage of all households: {zero_income_count / total_households * 100:.2f}%")
print("="*70)


HOUSEHOLDS WITH $0 INCOME
Household count: 174,584
Percentage of all households: 12.28%


In [10]:
# Income distribution by Census-style brackets
hh_income = np.array(sim.calculate("household_net_income", period=2025))
weights = np.array(sim.calculate("household_weight", period=2025))
total_households = weights.sum()

income_brackets = [
    (None, 10000, "Less than $10,000"),
    (10000, 15000, "$10,000 to $14,999"),
    (15000, 20000, "$15,000 to $19,999"),
    (20000, 25000, "$20,000 to $24,999"),
    (25000, 30000, "$25,000 to $29,999"),
    (30000, 35000, "$30,000 to $34,999"),
    (35000, 40000, "$35,000 to $39,999"),
    (40000, 45000, "$40,000 to $44,999"),
    (45000, 50000, "$45,000 to $49,999"),
    (50000, 60000, "$50,000 to $59,999"),
    (60000, 75000, "$60,000 to $74,999"),
    (75000, 100000, "$75,000 to $99,999"),
    (100000, 125000, "$100,000 to $124,999"),
    (125000, 150000, "$125,000 to $149,999"),
    (150000, 200000, "$150,000 to $199,999"),
    (200000, None, "$200,000 or more"),
]

bracket_data = []
for lower, upper, label in income_brackets:
    if lower is None:
        mask = hh_income < upper
    elif upper is None:
        mask = hh_income >= lower
    else:
        mask = (hh_income >= lower) & (hh_income < upper)
    
    count = weights[mask].sum()
    pct = count / total_households * 100
    
    bracket_data.append({
        "Income Level": label,
        "Households": int(round(count)),
        "Percent": f"{pct:.0f}%"
    })

income_df = pd.DataFrame(bracket_data)

print("\n" + "="*60)
print("Oregon Income Levels or Income Distribution")
print("="*60)
print(f"{'Households':<25} {int(round(total_households)):>12,}    100%")
print("-"*60)
for _, row in income_df.iterrows():
    print(f"{row['Income Level']:<25} {row['Households']:>12,}    {row['Percent']:>3}")
print("="*60)


Oregon Income Levels or Income Distribution
Households                   1,421,300    100%
------------------------------------------------------------
Less than $10,000              143,114    10%
$10,000 to $14,999              64,845     5%
$15,000 to $19,999              79,061     6%
$20,000 to $24,999              81,193     6%
$25,000 to $29,999              84,881     6%
$30,000 to $34,999              90,474     6%
$35,000 to $39,999              76,529     5%
$40,000 to $44,999              86,980     6%
$45,000 to $49,999              66,037     5%
$50,000 to $59,999             141,424    10%
$60,000 to $74,999             149,455    11%
$75,000 to $99,999             119,919     8%
$100,000 to $124,999            67,285     5%
$125,000 to $149,999            34,898     2%
$150,000 to $199,999            46,917     3%
$200,000 or more                88,287     6%
